# Corrected whole-session locomotor spectral analysis

This notebook implements the corrected first-stage analysis for the AIR Wheel spectral project.

## Main corrections

1. True path speed is derived from native `dist_path_cm`.
2. Nonoverlapping frequency-bin sums make relative band powers sum to approximately 1.
3. Fast and slow PSDs are calculated separately:
   - fast: 10-s Welch windows, 0.1–20 Hz;
   - slow: 60-s Welch windows, 0.02–5 Hz.
4. Immobility uses total encoder-count change rather than count range.
5. Artifact QC flags corrupted sessions but retains them in the full table.
6. Summaries and PSD curves weight animals equally.
7. Experience effects use one slope per animal-phase plus leave-one-animal-out checks.
8. All plots are saved as PNG files.

Save the Python module as:

```text
src/proc/whole_session_spectral_batch.py
```

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

In [ ]:
def find_repo_root(start_path=None):
    start_path = Path.cwd() if start_path is None else Path(start_path)
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise RuntimeError("Could not find the repository root containing src/.")


repo_root = find_repo_root()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)

In [ ]:
import src.utils.config as config
import src.utils.data_io as dio

from src.proc.whole_session_spectral_batch import (
    WholeSessionSpectralConfig,
    build_whole_session_spectral_tables,
    save_whole_session_spectral_tables,
)

print("RAW_BASE:", config.RAW_BASE)
print("PROC_BASE:", config.PROC_BASE)

## 1. Build `cc_data`

In [ ]:
data_root = config.RAW_BASE
cc_data = dio.build_classical_conditioning_dict(data_root)
print("Top-level cc_data entries:", len(cc_data))

## 2. Configure corrected processing and QC

In [ ]:
spectral_cfg = WholeSessionSpectralConfig(
    analysis_hz=100.0,
    derivative_window_ms=20.0,
    derivative_polyorder=3,
    speed_lowpass_hz=20.0,
    filter_order=4,

    fast_welch_window_s=10.0,
    fast_welch_overlap_fraction=0.50,
    fast_fmin_hz=0.10,
    fast_fmax_hz=20.0,

    slow_welch_window_s=60.0,
    slow_welch_overlap_fraction=0.50,
    slow_fmin_hz=0.02,
    slow_fmax_hz=5.0,

    behavior_window_s=2.0,
    movement_threshold_cms=0.20,

    fast_spectral_slope_fmin_hz=1.0,
    fast_spectral_slope_fmax_hz=10.0,

    qc_min_native_stored_r=0.90,
    qc_max_native_stored_rmse_cms=1.0,
    qc_max_abs_speed_cms=100.0,
    qc_max_native_position_step_cm=2.0,
    qc_large_encoder_jump_counts=5.0,
)

spectral_cfg

## 3. Select valid phases

Restricting the batch to the three experimental phases prevents `unknown` entries in `cc_data` from producing irrelevant file-not-found errors.

In [ ]:
animals_to_process = None

phases_to_process = [
    "repeated_exposure",
    "air_training",
    "tone_air_training",
]

## 4. Run the corrected batch analysis

In [ ]:
(
    whole_session_df,
    whole_session_psd_df,
    whole_session_errors_df,
) = build_whole_session_spectral_tables(
    cc_data=cc_data,
    config=spectral_cfg,
    root=None,
    animals=animals_to_process,
    phases=phases_to_process,
    verbose=True,
)

print()
print("Sessions processed:", len(whole_session_df))
print("Animals:", whole_session_df["animal"].nunique())
print("PSD rows:", len(whole_session_psd_df))
print("File-processing errors:", len(whole_session_errors_df))
print("QC failures:", int((~whole_session_df["qc_pass"]).sum()))

## 5. Save results and figures

In [ ]:
whole_session_output_dir = (
    Path(config.PROC_BASE)
    / "spectral_analysis"
    / "whole_session_corrected"
)

figure_dir = whole_session_output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

saved_paths = save_whole_session_spectral_tables(
    whole_session_df=whole_session_df,
    whole_session_psd_df=whole_session_psd_df,
    errors_df=whole_session_errors_df,
    output_dir=whole_session_output_dir,
)

print("Output directory:", whole_session_output_dir.resolve())
print("Figure directory:", figure_dir.resolve())

for name, path in saved_paths.items():
    print(f"{name}: {path}")

## 6. Coverage and processing errors

In [ ]:
coverage = (
    whole_session_df
    .groupby(["animal", "phase"], dropna=False)
    .agg(
        sessions=("date", "count"),
        qc_passed=("qc_pass", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

display(coverage)

if whole_session_errors_df.empty:
    print("No file-processing errors in the selected phases.")
else:
    display(whole_session_errors_df)

## 7. Artifact QC

The full table retains every processed session. `analysis_df` and `analysis_psd_df` contain only sessions passing QC.

In [ ]:
qc_columns = [
    "animal",
    "date",
    "phase",
    "qc_pass",
    "qc_reason",
    "native_vs_stored_speed_r",
    "native_vs_stored_speed_rmse_cms",
    "maximum_abs_speed_cms",
    "p999_abs_speed_cms",
    "maximum_native_position_step_cm",
    "maximum_encoder_count_step",
    "number_of_large_count_jumps",
]

failed_qc_df = whole_session_df.loc[~whole_session_df["qc_pass"]].copy()

print("Failed-QC sessions:", len(failed_qc_df))
display(failed_qc_df[qc_columns])

analysis_df = whole_session_df.loc[whole_session_df["qc_pass"]].copy()
analysis_psd_df = whole_session_psd_df.loc[
    whole_session_psd_df["qc_pass"]
].copy()

print("QC-passed sessions:", len(analysis_df))

## 8. Verify corrected band-power accounting

In [ ]:
band_sum_columns = [
    "signed_speed_fast_relative_band_power_sum",
    "path_speed_fast_relative_band_power_sum",
    "absolute_signed_speed_fast_relative_band_power_sum",
    "signed_speed_slow_relative_band_power_sum",
    "path_speed_slow_relative_band_power_sum",
    "absolute_signed_speed_slow_relative_band_power_sum",
]

display(analysis_df[band_sum_columns].describe())

## 9. Improved immobility noise-floor summary

In [ ]:
immobility_columns = [
    "animal",
    "date",
    "phase",
    "strict_immobile_2s_windows",
    "strict_immobile_2s_mean_abs_speed_p99_cms",
    "near_immobile_2s_windows",
    "near_immobile_2s_mean_abs_speed_p99_cms",
]

display(
    analysis_df[immobility_columns]
    .sort_values(["animal", "date"])
)

print(
    "Configured movement threshold:",
    spectral_cfg.movement_threshold_cms,
    "cm/s",
)

## 10. Validate true path-speed derivation

In [ ]:
path_validation_columns = [
    "animal",
    "date",
    "phase",
    "path_speed_source",
    "path_speed_negative_fraction_preclip",
    "mean_abs_signed_speed_cms",
    "mean_true_path_speed_cms",
    "path_distance_from_abs_signed_speed_cm",
    "path_distance_from_true_path_speed_cm",
]

display(analysis_df[path_validation_columns].head(20))

display(
    analysis_df[
        [
            "mean_abs_signed_speed_cms",
            "mean_true_path_speed_cms",
            "path_distance_from_abs_signed_speed_cm",
            "path_distance_from_true_path_speed_cm",
        ]
    ].corr()
)

## 11. Animal-weighted phase summaries

First compute one median per animal-phase. Then summarize those animal-level values across animals.

In [ ]:
numeric_columns = (
    analysis_df
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

animal_phase_df = (
    analysis_df
    .groupby(["animal", "phase"], as_index=False)[numeric_columns]
    .median()
)

animal_phase_path = (
    whole_session_output_dir
    / "animal_phase_median_features.csv"
)

animal_phase_df.to_csv(animal_phase_path, index=False)
print("Saved:", animal_phase_path)

In [ ]:
primary_features = [
    "fraction_moving_2s_windows",
    "mean_abs_signed_speed_cms",
    "mean_true_path_speed_cms",

    "signed_speed_slow_relative_power_cycle_0p02_0p05",
    "signed_speed_slow_relative_power_cycle_0p05_0p1",
    "signed_speed_slow_relative_power_very_slow_0p1_0p5",

    "signed_speed_fast_relative_power_slow_0p5_2",
    "signed_speed_fast_relative_power_intermediate_2_5",
    "signed_speed_fast_spectral_centroid_hz",
    "signed_speed_fast_spectral_entropy",

    "path_speed_fast_relative_power_slow_0p5_2",
    "path_speed_fast_relative_power_intermediate_2_5",
    "path_speed_fast_spectral_centroid_hz",
    "path_speed_fast_spectral_entropy",
]

animal_weighted_phase_summary = (
    animal_phase_df
    .groupby("phase")[primary_features]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

display(animal_weighted_phase_summary)

## 12. Phase-specific longitudinal plots

In [ ]:
def interpolated_group_median(phase_df, feature, grid=None):
    grid = np.linspace(0.0, 1.0, 101) if grid is None else grid
    interpolated = []

    for animal, animal_df in phase_df.groupby("animal"):
        animal_df = (
            animal_df
            .sort_values("phase_progress_0_1")
            .dropna(subset=["phase_progress_0_1", feature])
        )

        if len(animal_df) < 2:
            continue

        x = animal_df["phase_progress_0_1"].to_numpy()
        y = animal_df[feature].to_numpy()
        unique_x, unique_index = np.unique(x, return_index=True)
        unique_y = y[unique_index]

        if len(unique_x) >= 2:
            interpolated.append(np.interp(grid, unique_x, unique_y))

    if not interpolated:
        return grid, np.full_like(grid, np.nan)

    return grid, np.nanmedian(np.vstack(interpolated), axis=0)


def plot_feature_by_phase_progress(session_df, feature, ylabel=None):
    saved = []

    for phase, phase_df in session_df.groupby("phase"):
        fig, ax = plt.subplots(figsize=(9, 5))

        for animal, animal_df in phase_df.groupby("animal"):
            animal_df = animal_df.sort_values("phase_progress_0_1")
            ax.plot(
                animal_df["phase_progress_0_1"],
                animal_df[feature],
                marker="o",
                alpha=0.65,
                label=animal,
            )

        grid, group_median = interpolated_group_median(phase_df, feature)
        ax.plot(grid, group_median, linewidth=3, label="Across-animal median")

        ax.set_xlabel("Within-phase progress (0–1)")
        ax.set_ylabel(ylabel if ylabel is not None else feature)
        ax.set_title(f"{phase}: {feature}")
        ax.legend(title="Animal", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()

        path = figure_dir / f"{phase}_{feature}.png"
        fig.savefig(path, dpi=200)
        saved.append(path)
        plt.show()
        plt.close(fig)

    return saved

In [ ]:
features_to_plot = [
    ("fraction_moving_2s_windows", "Fraction of 2-s windows moving"),
    ("mean_abs_signed_speed_cms", "Mean absolute signed speed (cm/s)"),
    (
        "signed_speed_slow_relative_power_cycle_0p02_0p05",
        "Relative signed-speed power, 0.02–0.05 Hz",
    ),
    (
        "signed_speed_slow_relative_power_very_slow_0p1_0p5",
        "Relative signed-speed power, 0.1–0.5 Hz",
    ),
    (
        "signed_speed_fast_relative_power_slow_0p5_2",
        "Relative signed-speed power, 0.5–2 Hz",
    ),
    (
        "signed_speed_fast_relative_power_intermediate_2_5",
        "Relative signed-speed power, 2–5 Hz",
    ),
    (
        "signed_speed_fast_spectral_centroid_hz",
        "Signed-speed spectral centroid (Hz)",
    ),
    (
        "signed_speed_fast_spectral_entropy",
        "Signed-speed spectral entropy",
    ),
]

for feature, ylabel in features_to_plot:
    plot_feature_by_phase_progress(
        analysis_df,
        feature=feature,
        ylabel=ylabel,
    )

## 13. Equal-animal PSD summaries

The PSD is first summarized within each animal-phase and only then summarized across animals.

In [ ]:
animal_phase_psd_df = (
    analysis_psd_df
    .groupby(
        ["animal", "phase", "signal", "resolution", "frequency_hz"],
        as_index=False,
    )
    .agg(
        power=("power", "median"),
        relative_power_density=("relative_power_density", "median"),
    )
)

animal_phase_psd_path = (
    whole_session_output_dir
    / "animal_phase_psd_median.csv"
)

animal_phase_psd_df.to_csv(animal_phase_psd_path, index=False)
print("Saved:", animal_phase_psd_path)

In [ ]:
def plot_equal_animal_phase_psd(
    animal_phase_psd,
    signal,
    resolution,
    relative=False,
):
    value_column = "relative_power_density" if relative else "power"

    subset = animal_phase_psd.loc[
        (animal_phase_psd["signal"] == signal)
        & (animal_phase_psd["resolution"] == resolution)
    ].copy()

    saved = []

    for phase, phase_df in subset.groupby("phase"):
        fig, ax = plt.subplots(figsize=(9, 5))

        for animal, animal_df in phase_df.groupby("animal"):
            animal_df = animal_df.sort_values("frequency_hz")
            plot_method = ax.plot if relative else ax.semilogy
            plot_method(
                animal_df["frequency_hz"],
                animal_df[value_column],
                alpha=0.55,
                label=animal,
            )

        group_median = (
            phase_df
            .groupby("frequency_hz", as_index=False)[value_column]
            .median()
        )

        plot_method = ax.plot if relative else ax.semilogy
        plot_method(
            group_median["frequency_hz"],
            group_median[value_column],
            linewidth=3,
            label="Across-animal median",
        )

        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel(
            "Relative power density"
            if relative
            else "PSD [(cm/s)²/Hz]"
        )
        ax.set_title(f"{phase}: {signal}, {resolution} PSD")
        ax.legend(title="Animal", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()

        path = figure_dir / (
            f"{phase}_{signal}_{resolution}_"
            f"{'relative' if relative else 'absolute'}_psd.png"
        )
        fig.savefig(path, dpi=200)
        saved.append(path)
        plt.show()
        plt.close(fig)

    return saved

In [ ]:
for signal in ["signed_speed", "path_speed"]:
    for resolution in ["slow", "fast"]:
        plot_equal_animal_phase_psd(
            animal_phase_psd_df,
            signal=signal,
            resolution=resolution,
            relative=False,
        )
        plot_equal_animal_phase_psd(
            animal_phase_psd_df,
            signal=signal,
            resolution=resolution,
            relative=True,
        )

## 14. Animal-level within-phase slopes

Each animal contributes one slope per phase and feature. This is the primary whole-session experience screen.

In [ ]:
def calculate_animal_phase_slopes(session_df, features):
    rows = []

    for (animal, phase), group_df in session_df.groupby(["animal", "phase"]):
        group_df = group_df.sort_values("phase_progress_0_1")
        x = group_df["phase_progress_0_1"].to_numpy(dtype=float)

        for feature in features:
            y = group_df[feature].to_numpy(dtype=float)
            finite = np.isfinite(x) & np.isfinite(y)

            if np.count_nonzero(finite) < 3:
                continue

            slope, intercept = np.polyfit(x[finite], y[finite], deg=1)
            rows.append(
                {
                    "animal": animal,
                    "phase": phase,
                    "feature": feature,
                    "n_sessions": int(np.count_nonzero(finite)),
                    "slope_per_phase_progress": slope,
                    "intercept": intercept,
                }
            )

    return pd.DataFrame(rows)

In [ ]:
slope_features = [
    "fraction_moving_2s_windows",
    "mean_abs_signed_speed_cms",
    "signed_speed_slow_relative_power_cycle_0p02_0p05",
    "signed_speed_slow_relative_power_cycle_0p05_0p1",
    "signed_speed_slow_relative_power_very_slow_0p1_0p5",
    "signed_speed_fast_relative_power_slow_0p5_2",
    "signed_speed_fast_relative_power_intermediate_2_5",
    "signed_speed_fast_spectral_centroid_hz",
    "signed_speed_fast_spectral_entropy",
    "path_speed_fast_relative_power_slow_0p5_2",
    "path_speed_fast_relative_power_intermediate_2_5",
    "path_speed_fast_spectral_centroid_hz",
    "path_speed_fast_spectral_entropy",
]

animal_phase_slopes_df = calculate_animal_phase_slopes(
    analysis_df,
    slope_features,
)

slope_path = whole_session_output_dir / "animal_phase_feature_slopes.csv"
animal_phase_slopes_df.to_csv(slope_path, index=False)
print("Saved:", slope_path)
display(animal_phase_slopes_df.head())

In [ ]:
slope_consistency_df = (
    animal_phase_slopes_df
    .groupby(["phase", "feature"], as_index=False)
    .agg(
        n_animals=("animal", "nunique"),
        median_slope=("slope_per_phase_progress", "median"),
        minimum_slope=("slope_per_phase_progress", "min"),
        maximum_slope=("slope_per_phase_progress", "max"),
        positive_animals=(
            "slope_per_phase_progress",
            lambda values: int(np.sum(np.asarray(values) > 0)),
        ),
        negative_animals=(
            "slope_per_phase_progress",
            lambda values: int(np.sum(np.asarray(values) < 0)),
        ),
    )
)

slope_consistency_path = (
    whole_session_output_dir
    / "animal_phase_slope_consistency.csv"
)

slope_consistency_df.to_csv(slope_consistency_path, index=False)
print("Saved:", slope_consistency_path)
display(slope_consistency_df.sort_values(["feature", "phase"]))

## 15. Plot animal-level slopes

In [ ]:
def plot_animal_slopes(slopes_df, feature):
    plot_df = slopes_df.loc[slopes_df["feature"] == feature].copy()
    phases = [
        phase
        for phase in [
            "repeated_exposure",
            "air_training",
            "tone_air_training",
        ]
        if phase in plot_df["phase"].unique()
    ]

    fig, ax = plt.subplots(figsize=(9, 5))

    for phase_index, phase in enumerate(phases):
        phase_df = plot_df.loc[plot_df["phase"] == phase].sort_values("animal")
        offsets = np.linspace(-0.12, 0.12, len(phase_df))

        ax.scatter(
            phase_index + offsets,
            phase_df["slope_per_phase_progress"],
        )

        median_slope = phase_df["slope_per_phase_progress"].median()
        ax.plot(
            [phase_index - 0.20, phase_index + 0.20],
            [median_slope, median_slope],
            linewidth=3,
        )

        for offset, (_, row) in zip(offsets, phase_df.iterrows()):
            ax.text(
                phase_index + offset,
                row["slope_per_phase_progress"],
                row["animal"],
                fontsize=8,
                ha="center",
                va="bottom",
            )

    ax.axhline(0, linewidth=1)
    ax.set_xticks(range(len(phases)))
    ax.set_xticklabels(phases)
    ax.set_ylabel("Slope over within-phase progress")
    ax.set_title(feature)
    fig.tight_layout()

    path = figure_dir / f"animal_slopes_{feature}.png"
    fig.savefig(path, dpi=200)
    plt.show()
    plt.close(fig)
    return path


for feature in slope_features:
    plot_animal_slopes(animal_phase_slopes_df, feature)

## 16. Leave-one-animal-out slope sensitivity

In [ ]:
def leave_one_animal_out_slope_summary(slopes_df):
    rows = []
    all_animals = sorted(slopes_df["animal"].unique())

    for (phase, feature), group_df in slopes_df.groupby(["phase", "feature"]):
        full_median = group_df["slope_per_phase_progress"].median()

        for excluded_animal in all_animals:
            retained = group_df.loc[group_df["animal"] != excluded_animal]
            if retained.empty:
                continue

            rows.append(
                {
                    "phase": phase,
                    "feature": feature,
                    "excluded_animal": excluded_animal,
                    "full_sample_median_slope": full_median,
                    "leave_one_out_median_slope": retained[
                        "slope_per_phase_progress"
                    ].median(),
                    "leave_one_out_minimum_slope": retained[
                        "slope_per_phase_progress"
                    ].min(),
                    "leave_one_out_maximum_slope": retained[
                        "slope_per_phase_progress"
                    ].max(),
                }
            )

    return pd.DataFrame(rows)


leave_one_out_df = leave_one_animal_out_slope_summary(
    animal_phase_slopes_df
)

leave_one_out_path = (
    whole_session_output_dir
    / "leave_one_animal_out_slope_sensitivity.csv"
)

leave_one_out_df.to_csv(leave_one_out_path, index=False)
print("Saved:", leave_one_out_path)
display(leave_one_out_df.head())

## 17. Spectral outcomes versus movement amount

In [ ]:
correlation_columns = [
    "mean_abs_signed_speed_cms",
    "mean_true_path_speed_cms",
    "fraction_moving_2s_windows",
    "signed_speed_slow_relative_power_cycle_0p02_0p05",
    "signed_speed_slow_relative_power_cycle_0p05_0p1",
    "signed_speed_slow_relative_power_very_slow_0p1_0p5",
    "signed_speed_fast_total_power",
    "signed_speed_fast_relative_power_slow_0p5_2",
    "signed_speed_fast_relative_power_intermediate_2_5",
    "signed_speed_fast_spectral_centroid_hz",
    "signed_speed_fast_spectral_entropy",
    "path_speed_fast_relative_power_slow_0p5_2",
    "path_speed_fast_relative_power_intermediate_2_5",
    "path_speed_fast_spectral_centroid_hz",
    "path_speed_fast_spectral_entropy",
]

display(analysis_df[correlation_columns].corr().round(3))

In [ ]:
def plot_feature_against_speed(
    session_df,
    feature,
    speed_feature="mean_abs_signed_speed_cms",
):
    fig, ax = plt.subplots(figsize=(7, 5))

    for animal, animal_df in session_df.groupby("animal"):
        ax.scatter(
            animal_df[speed_feature],
            animal_df[feature],
            label=animal,
            alpha=0.70,
        )

    ax.set_xlabel(speed_feature)
    ax.set_ylabel(feature)
    ax.set_title(f"{feature} versus {speed_feature}")
    ax.legend(title="Animal", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()

    path = figure_dir / f"{feature}_versus_{speed_feature}.png"
    fig.savefig(path, dpi=200)
    plt.show()
    plt.close(fig)
    return path


for feature in [
    "signed_speed_slow_relative_power_cycle_0p02_0p05",
    "signed_speed_slow_relative_power_very_slow_0p1_0p5",
    "signed_speed_fast_relative_power_slow_0p5_2",
    "signed_speed_fast_relative_power_intermediate_2_5",
    "signed_speed_fast_spectral_centroid_hz",
    "signed_speed_fast_spectral_entropy",
]:
    plot_feature_against_speed(analysis_df, feature)

## 18. Files produced

Primary outputs:

```text
whole_session_spectral_features_corrected.csv
whole_session_psd_long_corrected.csv
whole_session_spectral_errors_corrected.csv
whole_session_qc_failed_sessions.csv
```

Animal-level summaries:

```text
animal_phase_median_features.csv
animal_phase_psd_median.csv
animal_phase_feature_slopes.csv
animal_phase_slope_consistency.csv
leave_one_animal_out_slope_sensitivity.csv
```

Figures:

```text
<PROC_BASE>/spectral_analysis/whole_session_corrected/figures/
```

## 19. Reload saved tables without rerunning native-rate processing

In [ ]:
# whole_session_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_spectral_features_corrected.csv"
# )
#
# whole_session_psd_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_psd_long_corrected.csv"
# )
#
# whole_session_errors_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_spectral_errors_corrected.csv"
# )
#
# analysis_df = whole_session_df.loc[whole_session_df["qc_pass"]].copy()
# analysis_psd_df = whole_session_psd_df.loc[
#     whole_session_psd_df["qc_pass"]
# ].copy()